# The Lorenz System
### Lesson 5, Section 2

In this section you will study Edward Lorenz's simplified model of atmospheric
convection — the original source of the *butterfly effect*:

$$\frac{dx}{dt} = \sigma(y - x), \qquad
  \frac{dy}{dt} = \rho\,x - y - xz, \qquad
  \frac{dz}{dt} = xy - bz$$

Classical chaotic parameters: $\sigma = 10$, $\rho = 28$, $b = 8/3$.

You will implement the equations, solve them numerically, visualise the strange
attractor, and demonstrate exponential divergence of nearby trajectories.


## 0 · Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('All imports OK ✓')


---
# Part 2 — The Lorenz System

Edward Lorenz's simplified model of atmospheric convection:

$$\frac{dx}{dt} = \sigma(y - x)$$
$$\frac{dy}{dt} = \rho\,x - y - x\,z$$
$$\frac{dz}{dt} = x\,y - b\,z$$

Classical chaotic parameters: $\sigma = 10$, $\rho = 28$, $b = 8/3$.


## 2.1 · Define the Lorenz equations


In [ ]:
def lorenz(t, state, sigma=10.0, rho=28.0, b=8/3):
    """Right-hand side of the Lorenz system."""
    x, y, z = state
    dx = sigma * (y - x)
    dy = rho * x - y - x * z
    dz = x * y - b * z
    return [dx, dy, dz]


### Independent test 3
At the origin $(0, 0, 0)$ all derivatives should be zero (it is an equilibrium point).


In [ ]:
# --- Test: equilibrium at origin ---
result = lorenz(0, [0.0, 0.0, 0.0])
print('Derivatives at origin:', result)
assert all(v == 0.0 for v in result)
print('Test passed ✓ The origin is an equilibrium.')


## 2.2 · Solve the Lorenz system


In [ ]:
T_end = 50.0
t_eval = np.linspace(0, T_end, 20000)

sol = solve_ivp(lorenz, (0, T_end), [1.0, 1.0, 1.0],
                t_eval=t_eval, max_step=0.01, rtol=1e-9)

print(f'Solver status: {"success" if sol.success else sol.message}')
print(f'Shape of solution: {sol.y.shape}')


## 2.3 · The strange attractor in 3D


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(sol.y[0], sol.y[1], sol.y[2], linewidth=0.4, color='steelblue')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Lorenz Strange Attractor')
plt.tight_layout()
plt.show()


## 2.4 · Time series of $x(t)$
The $x$ component switches unpredictably between positive and negative values — corresponding to the two "wings" of the attractor.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sol.t, sol.y[0], linewidth=0.5, color='steelblue')
ax.set_xlabel('Time')
ax.set_ylabel('x(t)')
ax.set_title('Lorenz System — x component over time')
plt.tight_layout()
plt.show()


## 2.5 · Sensitivity to initial conditions
Start two trajectories with a tiny difference ($10^{-9}$) and watch them diverge.


In [ ]:
ic_a = [1.0, 1.0, 1.0]
ic_b = [1.0, 1.0, 1.0 + 1e-9]

sol_a = solve_ivp(lorenz, (0, T_end), ic_a, t_eval=t_eval, max_step=0.01, rtol=1e-12)
sol_b = solve_ivp(lorenz, (0, T_end), ic_b, t_eval=t_eval, max_step=0.01, rtol=1e-12)

diff = np.sqrt(np.sum((sol_a.y - sol_b.y)**2, axis=0))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(sol_a.t, sol_a.y[0], linewidth=0.5, label='Trajectory A', color='steelblue')
axes[0].plot(sol_b.t, sol_b.y[0], linewidth=0.5, label='Trajectory B', color='firebrick', alpha=0.7)
axes[0].set_ylabel('x(t)')
axes[0].set_title('Two Lorenz trajectories with initial difference $10^{-9}$')
axes[0].legend()

axes[1].plot(sol_a.t, diff, linewidth=0.8, color='purple')
axes[1].set_yscale('log')
axes[1].set_ylabel('$||A - B||$')
axes[1].set_xlabel('Time')
axes[1].set_title('Distance between the two trajectories')

plt.tight_layout()
plt.show()

print('Observation: trajectories are indistinguishable at first, then diverge exponentially.')


### What to try
- Change `rho` to 15 (below the chaotic threshold) — does the attractor still look butterfly-shaped?
- Try `rho = 100` — what happens?
- Change the initial separation from $10^{-9}$ to $10^{-3}$ — how much sooner do the trajectories diverge?


---
## What to Try

1. Find the two non-origin equilibrium points of the Lorenz system.
   Hint: set all derivatives to zero and solve for $x, y, z$.
2. Plot all three components $x(t)$, $y(t)$, $z(t)$ together on the same axes.
   Which component is most regular?
3. Change `rho` to `15` (below the chaotic threshold) — does the attractor
   still look butterfly-shaped?
4. Try `rho = 100` — what happens to the trajectory?
5. Change the initial separation from $10^{-9}$ to $10^{-3}$ — how much
   sooner do the trajectories diverge?


---
## Mini-Projects

### Project: Parameter Sweep
Vary $\rho$ from 0 to 50 and classify the long-term behaviour (fixed point,
periodic, chaotic) for each value. Plot a 'bifurcation-like' diagram for the
Lorenz system — e.g. record the local maxima of $z(t)$ after discarding
transients.


